<a href="https://colab.research.google.com/github/grumpyoldking/COMP3710/blob/main/DAWNbench.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import sys, os, time, argparse, random
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torch.amp import autocast, GradScaler  # New AMP API

# ----------------- Setup -----------------
def seed_all(seed=42):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True  # fastest conv algos for fixed 32x32

# Optional: allow TF32 on Ampere+ for extra throughput
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

# ----------------- Model -----------------
class FastCIFAR10Net(nn.Module):
    """
    Tiny, fast CNN w/ 3 stages and global avg pool -> Linear(10).
    32x32 -> (stride-2) 16x16 -> (stride-2) 8x8 -> GAP -> FC
    """
    def __init__(self, num_classes=10, drop=0.0):
        super().__init__()

        def conv_bn_relu(cin, cout, k=3, s=1, p=1):
            return nn.Sequential(
                nn.Conv2d(cin, cout, k, stride=s, padding=p, bias=False),
                nn.BatchNorm2d(cout),
                nn.ReLU(inplace=True),
            )

        self.stem = conv_bn_relu(3, 64, 3, 1, 1)
        self.block1 = nn.Sequential(
            conv_bn_relu(64, 64),
            conv_bn_relu(64, 64),
            conv_bn_relu(64, 64, s=2),   # 32->16
        )
        self.block2 = nn.Sequential(
            conv_bn_relu(64, 128),
            conv_bn_relu(128, 128),
            conv_bn_relu(128, 128, s=2), # 16->8
        )
        self.block3 = nn.Sequential(
            conv_bn_relu(128, 256),
            conv_bn_relu(256, 256),
        )
        self.dropout = nn.Dropout(drop) if drop > 0 else nn.Identity()
        self.head = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.stem(x)            # [B, 64, 32, 32]
        x = self.block1(x)          # [B, 64, 16, 16]
        x = self.block2(x)          # [B,128,  8,  8]
        x = self.block3(x)          # [B,256,  8,  8]
        x = self.dropout(x)
        x = F.adaptive_avg_pool2d(x, 1).flatten(1)  # [B,256]
        return self.head(x)                          # [B,10]

# ----------------- Loss -----------------
class SmoothCELoss(nn.Module):
    def __init__(self, smoothing=0.1):
        super().__init__()
        assert 0.0 <= smoothing < 1.0
        self.smoothing = smoothing

    def forward(self, logits, targets):
        n_class = logits.size(-1)
        log_probs = F.log_softmax(logits, dim=-1)
        with torch.no_grad():
            true_dist = torch.full_like(log_probs, self.smoothing / (n_class - 1))
            true_dist.scatter_(1, targets.unsqueeze(1), 1.0 - self.smoothing)
        return torch.mean(torch.sum(-true_dist * log_probs, dim=-1))

# ----------------- Data -----------------
def get_loaders(data_root, batch_size, num_workers):
    mean = (0.4914, 0.4822, 0.4465)
    std  = (0.2470, 0.2435, 0.2616)

    train_tf = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])
    test_tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])

    train_set = datasets.CIFAR10(data_root, train=True,  download=True, transform=train_tf)
    test_set  = datasets.CIFAR10(data_root, train=False, download=True, transform=test_tf)

    pin = torch.cuda.is_available()
    common = dict(batch_size=batch_size,
                  num_workers=num_workers,
                  pin_memory=pin,
                  persistent_workers=(num_workers > 0))
    if num_workers and num_workers > 0:
        common["prefetch_factor"] = 4

    train_loader = DataLoader(train_set, shuffle=True,  drop_last=True,  **common)
    test_loader  = DataLoader(test_set,  shuffle=False, drop_last=False, **common)
    return train_loader, test_loader

# ----------------- Eval -----------------
@torch.no_grad()
def evaluate(model, loader, device, channels_last=False):
    model.eval()
    correct, total = 0, 0
    for x, y in loader:
        if channels_last:
            x = x.to(device, non_blocking=True, memory_format=torch.channels_last)
        else:
            x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        pred = model(x).argmax(1)
        correct += (pred == y).sum().item()
        total += y.size(0)
    return correct / total

# ----------------- Train -----------------
def train(args):
    seed_all(args.seed)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    is_cuda = (device.type == 'cuda')
    channels_last = is_cuda

    train_loader, test_loader = get_loaders(args.data, args.batch_size, args.workers)

    model = FastCIFAR10Net(drop=args.dropout).to(device)
    if channels_last:
        model = model.to(memory_format=torch.channels_last)

    opt = torch.optim.SGD(model.parameters(), lr=args.lr,
                          momentum=0.9, weight_decay=5e-4, nesterov=True)

    steps_per_epoch = len(train_loader)
    total_steps = steps_per_epoch * args.epochs

    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=args.lr, total_steps=total_steps,
        pct_start=0.12, div_factor=25.0, final_div_factor=1e4,
        anneal_strategy='cos'
    )

    criterion = SmoothCELoss(args.label_smoothing)
    scaler = GradScaler(enabled=is_cuda)

    start_time = time.time()
    best_acc = 0.0

    for epoch in range(1, args.epochs + 1):
        model.train()
        for x, y in train_loader:
            if channels_last:
                x = x.to(device, non_blocking=True, memory_format=torch.channels_last)
            else:
                x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            opt.zero_grad(set_to_none=True)
            with autocast(device_type='cuda', dtype=torch.float16, enabled=is_cuda):
                logits = model(x)
                loss = criterion(logits, y)

            # (optional) clip for stability at high LR
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            scaler.step(opt)
            scaler.update()

            # OneCycleLR is a per-batch scheduler
            scheduler.step()

        acc = evaluate(model, test_loader, device, channels_last)
        best_acc = max(best_acc, acc)
        elapsed = time.time() - start_time
        print(f"Epoch {epoch:02d} | Acc {acc*100:.2f}% | Best {best_acc*100:.2f}% | Elapsed {elapsed:.1f}s")

        if acc * 100.0 >= args.target_acc:
            print(f"\nReached {args.target_acc:.2f}% in {elapsed:.2f} seconds. (Time-to-accuracy)")
            return

    elapsed = time.time() - start_time
    print(f"\nFinished {args.epochs} epochs. Best Acc {best_acc*100:.2f}% in {elapsed:.2f} seconds.")

# ----------------- CLI -----------------
def parse_args(argv=None):
    p = argparse.ArgumentParser()
    p.add_argument('--data', type=str, default='./data')
    p.add_argument('--batch-size', type=int, default=512)
    p.add_argument('--workers', type=int, default=(os.cpu_count() or 4))
    p.add_argument('--epochs', type=int, default=35)
    p.add_argument('--lr', type=float, default=0.4)  # OneCycle max LR
    p.add_argument('--label-smoothing', type=float, default=0.1)
    p.add_argument('--dropout', type=float, default=0.0)
    p.add_argument('--target-acc', type=float, default=94.0)
    p.add_argument('--seed', type=int, default=42)
    # Notebook/Colab: ignore foreign args like "-f ..."
    args, _ = p.parse_known_args(argv)
    return args

if __name__ == '__main__':
    # Jupyter/Colab friendly: pass [] so argparse ignores injected flags
    in_ipython = ('ipykernel' in sys.modules) or ('google.colab' in sys.modules)
    args = parse_args([] if in_ipython else None)
    train(args)


Epoch 01 | Acc 51.38% | Best 51.38% | Elapsed 3.8s
Epoch 02 | Acc 48.66% | Best 51.38% | Elapsed 7.1s
Epoch 03 | Acc 62.64% | Best 62.64% | Elapsed 10.5s
Epoch 04 | Acc 73.48% | Best 73.48% | Elapsed 13.8s
Epoch 05 | Acc 70.65% | Best 73.48% | Elapsed 17.0s
Epoch 06 | Acc 72.11% | Best 73.48% | Elapsed 20.5s
Epoch 07 | Acc 75.95% | Best 75.95% | Elapsed 23.7s
Epoch 08 | Acc 68.87% | Best 75.95% | Elapsed 27.0s
Epoch 09 | Acc 74.36% | Best 75.95% | Elapsed 30.2s
Epoch 10 | Acc 78.76% | Best 78.76% | Elapsed 33.5s
Epoch 11 | Acc 76.81% | Best 78.76% | Elapsed 36.8s
Epoch 12 | Acc 77.90% | Best 78.76% | Elapsed 40.0s
Epoch 13 | Acc 80.38% | Best 80.38% | Elapsed 43.3s
Epoch 14 | Acc 72.26% | Best 80.38% | Elapsed 46.6s
Epoch 15 | Acc 62.54% | Best 80.38% | Elapsed 50.0s
Epoch 16 | Acc 78.46% | Best 80.38% | Elapsed 53.2s
Epoch 17 | Acc 82.72% | Best 82.72% | Elapsed 56.4s
Epoch 18 | Acc 81.62% | Best 82.72% | Elapsed 59.6s
Epoch 19 | Acc 83.33% | Best 83.33% | Elapsed 62.9s
Epoch 20 | Acc